# Market Microstructure

`Brokerage.jl` is a simulated exchange: user accounts with cash and shares in an SQLite database, an order management system around a limit order book, and an HTTP interface. `TradingAgents.jl` provides agents that log in and trade through that interface in real time. This notebook runs a two-minute session with fundamental traders, zero-intelligence liquidity takers and a random market maker, then studies the trade tape.

**Run this notebook in a fresh Julia session**: the server, port 8080 and the tape are global, so a second session needs a restart.

In [ ]:
Pollis.packages("StatsBase", "Plots", "Brokerage" => "https://github.com/aaron-wheeler/Brokerage.jl", "TradingAgents" => "https://github.com/aaron-wheeler/TradingAgents.jl")

In [ ]:
using Brokerage, TradingAgents, Dates, Statistics, StatsBase, Plots

## Start the exchange

The account database goes in a temporary folder, because the installed package folder is read-only. Account ids 1 to 30 are reserved for market makers. The market opens 30 seconds from now, leaving time to create the accounts.

In [ ]:
N, F, minutes = 100, 100.0, 2
NF = div(N, 2)   # fundamental traders; the other N - NF are liquidity takers

DBFILE = joinpath(mktempdir(), "portfolios.sqlite")
AUTHFILE = "file://" * joinpath(dirname(pathof(Brokerage)), "../resources/authkeys.json")
Mapper.MM_COUNTER[] = 30
OMS.NUM_ASSETS[] = 1
OMS.PRICE_BUFFER_CAPACITY[] = 100
OMS.MARKET_OPEN_T[] = now() + Second(30)
OMS.MARKET_CLOSE_T[] = OMS.MARKET_OPEN_T[] + Minute(minutes)
OMS.init_LOB!(OMS.ob, [F], OMS.LP_order_vol, OMS.LP_cancel_vol, OMS.trade_volume_t, OMS.price_buffer)
server = @async Brokerage.run(DBFILE, AUTHFILE)
sleep(5)

## Start the agents

Each `*_run` call blocks until the market closes, so the agents run as tasks. Account ids are given out in order, so the liquidity takers start only after the fundamental traders' accounts exist. This cell waits until the close (about two and a half minutes).

In [ ]:
ft = @async FT_run(NF, 1, (init_cash_range = 5000.0:0.01:15000.0, init_shares_range = 50:1:150, prob_wait = 0.5, trade_freq = 1, num_ids = 30),
    (host_ip_address = "0.0.0.0", port = "8080", username = "Fundamental Trader", password = "value123"))
while Mapper.PORTFOLIO_COUNTER[] < 30 + NF
    sleep(0.5)
end
zt = @async ZT_run(N - NF, 1, (username = "Zero Trader", password = "zero123", init_cash_range = 5000.0:0.01:15000.0, init_shares_range = 50:1:150, prob_wait = 0.5, trade_freq = 1, num_MM = 30 + NF),
    (host_ip_address = "0.0.0.0", port = "8080"))
mm = @async RandomMM_run(1, (id = 1, eps_min = -0.5, eps_max = 0.5, inventory_limit = 3000, unit_trade_size = 15, trade_freq = 2), (cash = 0, z = 0),
    (host_ip_address = "0.0.0.0", port = "8080", username = "Market Maker", password = "liquidity000"))
foreach(wait, (ft, zt, mm))

## The trade tape

The exchange appends one row per **market order** to its tape. Trades from limit orders that cross the spread are not recorded, so the fundamental traders' trades are missing here.

In [ ]:
tp = copy(OMS.tick_last_prices)
q = OMS.tick_trading_volume   # signed shares: + buy, - sell
secs = Dates.value.(OMS.tick_time .- OMS.MARKET_OPEN_T[]) ./ 1000
println("Recorded trades: ", length(tp), ", shares traded: ", sum(abs.(q)))
println("Mean spread:     ", round(mean(OMS.tick_ask_prices .- OMS.tick_bid_prices); digits = 3))
plot(secs, tp; label = "Last Trade Price", xlabel = "Seconds Since Open", ylabel = "Price")

## Large orders move the price more

Unlike the one-share models, orders here have sizes. Compare the signed price move over the next 10 trades after the largest 5% of market orders with the move after the rest.

In [ ]:
sgn, sz = Int.(sign.(q)), abs.(q)
h = 10
move = [sgn[i] * (log(tp[i + h]) - log(tp[i])) for i in 1:length(tp) - h]
big = sz[1:length(move)] .>= quantile(sz, 0.95)
println("Largest 5%: ", round(1e4 * mean(move[big]); digits = 1), " bp, others: ", round(1e4 * mean(move[.!big]); digits = 1), " bp")

## The final book and the accounts

In [ ]:
d = Client.getBookDepth(1)
bar(Float64.(d[:BID][:price]), cumsum(Float64.(d[:BID][:volume])); label = "Bids", alpha = 0.6, xlabel = "Price", ylabel = "Cumulative Shares")
bar!(Float64.(d[:ASK][:price]), cumsum(Float64.(d[:ASK][:volume])); label = "Asks", alpha = 0.6)

In [ ]:
for id in (31, 31 + NF)   # the first fundamental trader and the first liquidity taker
    println("Account ", id, ": cash ", Client.getCash(id), ", holdings ", Client.getHoldings(id))
end

## Exercises

1. Compute the excess kurtosis and the lag-1 autocorrelation of `diff(log.(tp))`. Compare with the Trading Agents notebook.
2. Restart Julia and run a session with only liquidity takers (`NF = 0`, and skip `FT_run`). What happens to the price and the spread?
3. Plot `cumsum(q)`, the net order flow. Does the price follow it?
4. Read the Trading Agents wiki page and replace `FT_run` and `ZT_run` with their parallel successors `PLP_run` and `PLT_run`.